<a href="https://colab.research.google.com/github/Jakins-web/Predictive-Gas_Leakage-EdgeAI/blob/main/Gas_Leakage_EdgeAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv('AirQualityUCI.csv', sep = ';', decimal = ',' )
df.head()


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


In [16]:
# 1. Drop the completely empty columns (15 and 16)
# axis=1 means columns, how='all' means only drop if every value is NaN
df = df.dropna(axis=1, how='all')

# 2. Trim the trailing empty rows
# (This brings the 9471 rows down to the valid 9357 rows)
df = df.dropna(axis=0, how='all')

# 3. Handle the "Engineering Error" code (-200)
# In this dataset, -200 is a placeholder for a failed sensor.
# If we leave it, the Gradient Descent will think the air quality is -200 (impossible!)
df.replace(-200, np.nan, inplace=True)

# 4. Final check: How many missing values are left per sensor?
print("Missing values per sensor:")
df.isnull().sum().sum()

Missing values per sensor:


np.int64(16701)

In [17]:
df.interpolate(method='linear', inplace=True)
# Apply linear interpolation
df.interpolate(method='linear', inplace=True)

# Important: If the VERY FIRST row was missing,
# interpolate has no "starting point." Use bfill to fix that.
#df.fillna(method='bfill', inplace=True)

/tmp/ipykernel_2239/2488466435.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df.interpolate(method='linear', inplace=True)
/tmp/ipykernel_2239/2488466435.py:3: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df.interpolate(method='linear', inplace=True)


In [18]:
df.isnull().sum().sum()

np.int64(0)

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# 1. Prepare Data (Using the columns you identified yesterday)
X = df[['PT08.S2(NMHC)', 'PT08.S1(CO)']]
y = df['C6H6(GT)']

# 2. THE SPLIT: Hide 20% of the data for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. THE MODEL: No more manual loops!
model = LinearRegression()
model.fit(X_train, y_train) # This runs the gradient descent internally

# 4. THE TEST: Predict on the data the model has NEVER seen
y_pred = model.predict(X_test)

# 5. THE VERDICT: How good are we?
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

R2 Score: 0.9640
Mean Error (RMSE): 1.4396


In [20]:
# To Create a DataFrame to compare the first 10 results
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
comparison_df['Error'] = comparison_df['Actual'] - comparison_df['Predicted']

print(comparison_df.head(10))

      Actual  Predicted     Error
2973     6.4   7.070265 -0.670265
3396     4.9   5.156571 -0.256571
4372    14.6  15.542845 -0.942845
6025    14.5  15.353295 -0.853295
7960     4.9   5.176628 -0.276628
5263     5.4   5.851645 -0.451645
8331     3.8   3.572333  0.227667
1745    11.6  12.826254 -1.226254
5148     7.2   8.057947 -0.857947
2111     4.9   5.016012 -0.116012


In [21]:
# Create a new column: 1 if Benzene > 10, else 0
df['Safety_Status'] = (df['C6H6(GT)'] > 10).astype(int)

print("Safety Status Counts:")
print(df['Safety_Status'].value_counts())

Safety Status Counts:
Safety_Status
0    5562
1    3795
Name: count, dtype: int64


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Prepare Data
# We use our favorite sensors to predict the binary Safety_Status
X = df[['PT08.S2(NMHC)', 'PT08.S1(CO)']]
y = df['Safety_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train the "Soft Logic Gate"
log_model = LogisticRegression()
log_model.fit(X_train, y_train)

# 3. Predict
y_pred = log_model.predict(X_test)

# 4. Evaluate Accuracy
print(f"Classification Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Classification Accuracy: 0.9984


In [23]:
lg_comparison = pd.DataFrame({'Actual': y_test, 'L_reg_Predicted': y_pred})
lg_comparison['Error'] = lg_comparison['Actual'] - lg_comparison['L_reg_Predicted']

print(f"L_reg R2: {r2_score(y_test, y_pred):.4f}")
print(lg_comparison.head(10))

L_reg R2: 0.9933
      Actual  L_reg_Predicted  Error
2973       0                0      0
3396       0                0      0
4372       1                1      0
6025       1                1      0
7960       0                0      0
5263       0                0      0
8331       0                0      0
1745       1                1      0
5148       0                0      0
2111       0                0      0


In [24]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[1128    1]
 [   2  741]]


In [25]:
# Instead of predict(), we get the raw probabilities
# [Prob of Safe, Prob of Dangerous]
y_probs = log_model.predict_proba(X_test)[:, 1]

# We set a custom "Engineering Safety Threshold"
# If the probability of danger is > 10%, sound the alarm!
threshold = 0.1
y_pred_safe = (y_probs > threshold).astype(int)

# Now check the Confusion Matrix again
from sklearn.metrics import confusion_matrix
print(f"Safety-First Confusion Matrix (Threshold {threshold}):")
print(confusion_matrix(y_test, y_pred_safe))

Safety-First Confusion Matrix (Threshold 0.1):
[[1112   17]
 [   1  742]]


In [26]:
from sklearn.ensemble import RandomForestRegressor

# We use the committee of trees to find non-linear patterns
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Let's see the new comparison
rf_comparison = pd.DataFrame({'Actual': y_test, 'RF_Predicted': rf_pred})
rf_comparison['Error'] = rf_comparison['Actual'] - rf_comparison['RF_Predicted']

print(f"Random Forest R2: {r2_score(y_test, rf_pred):.4f}")
print(rf_comparison.head(10))

Random Forest R2: 0.9959
      Actual  RF_Predicted  Error
2973       0           0.0    0.0
3396       0           0.0    0.0
4372       1           1.0    0.0
6025       1           1.0    0.0
7960       0           0.0    0.0
5263       0           0.0    0.0
8331       0           0.0    0.0
1745       1           1.0    0.0
5148       0           0.0    0.0
2111       0           0.0    0.0
